# MutPred2 score coverage vs. extracted variants

Checks every sheet of `extractedVariants.xlsx` against `MutPred2_scores.tsv` and reports,
per disease sheet and for `nonTarget`, how many variants have **no MutPred2 score**.

The expectation is that coverage is 100%. Where it is not, the last section works out *why*
by walking back up the pipeline to the VEP annotation step.

All paths come from `ExTSP.config`, so this notebook keeps working if files are moved.

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

from ExTSP.config import (
    MutPred2_filtered_file,   # -> filtered/MutPred2_scores.tsv
    ExtractedVariants_file,   # -> filtered/extractedVariants.xlsx
    VariantAnnotations_file,  # -> filtered/variant_annotations.tsv
    FILTERED_DIR,
    DATA_DIR,
)

pd.set_option("display.width", 160)

# Everything this notebook writes goes here, kept separate from the curated data.
AI_GENERATED_DIR = Path(str(DATA_DIR)) / "AI_generated"
AI_GENERATED_DIR.mkdir(exist_ok=True)

for f in (MutPred2_filtered_file, ExtractedVariants_file, VariantAnnotations_file):
    print(f"{'OK ' if f.exists() else 'MISSING'}  {f}")
print(f"outputs -> {AI_GENERATED_DIR}")

## 1. Load the MutPred2 scores

The join key is `Variant` (`chr_pos_ref/alt`). `MutPred2_scores.tsv` is at
variant-transcript resolution, so one variant can appear on several rows; we reduce it
to the set of variants that carry at least one non-null score.

In [ ]:
import pandas as pd

from ExTSP.config import MutPred2_filtered_file

mp2 = pd.read_csv(MutPred2_filtered_file, sep="\t")

print("columns          :", list(mp2.columns))
print("rows             :", len(mp2))
print("unique variants  :", mp2["Variant"].nunique())
print("unique ENST      :", mp2["ENST"].nunique())
print("null MutPred2    :", mp2["MutPred2"].isna().sum())

# Variants that actually carry a score.
scored_variants = set(mp2.loc[mp2["MutPred2"].notna(), "Variant"].dropna().unique())
print("scored variants  :", len(scored_variants))

## 2. Load every sheet of extractedVariants.xlsx

Each sheet carries a `Variant` column in the same `chr_pos_ref/alt` format, so no key
normalisation is needed. (Section 5 verifies that assumption rather than trusting it.)

In [ ]:
import pandas as pd

from ExTSP.config import ExtractedVariants_file

xl = pd.ExcelFile(ExtractedVariants_file)
print("sheets:", xl.sheet_names)

sheets = {name: xl.parse(name) for name in xl.sheet_names}

for name, df in sheets.items():
    has_variant = "Variant" in df.columns
    print(f"  {name:<14} rows={len(df):>7}  cols={len(df.columns):>3}  Variant column: {has_variant}")

## 3. Coverage per sheet

`missing` = variants in the sheet with no MutPred2 score anywhere in `MutPred2_scores.tsv`.

In [ ]:
import numpy as np
import pandas as pd

def coverage(df, scored):
    """Coverage of one sheet's unique variants against the scored set."""
    variants = set(df["Variant"].dropna().unique())
    missing = variants - scored
    return {
        "rows": len(df),
        "unique_variants": len(variants),
        "found": len(variants) - len(missing),
        "missing": len(missing),
        "pct_missing": round(100 * len(missing) / len(variants), 2) if variants else np.nan,
    }


missing_by_sheet = {}
records = {}

for name, df in sheets.items():
    if "Variant" not in df.columns:
        print(f"skipping {name}: no Variant column")
        continue
    records[name] = coverage(df, scored_variants)
    missing_by_sheet[name] = set(df["Variant"].dropna().unique()) - scored_variants

summary = pd.DataFrame(records).T
summary.index.name = "sheet"
summary

### The headline numbers

Disease sheets and `nonTarget` called out explicitly, as requested.

In [ ]:
DISEASES = ["CM", "PCD", "PH", "PKD", "ASD"]

print("Variants with NO MutPred2 score")
print("-" * 52)
for d in DISEASES:
    if d in records:
        r = records[d]
        print(f"  {d:<10} {r['missing']:>6} of {r['unique_variants']:>6}  ({r['pct_missing']:>5.1f}%)")

print("-" * 52)
for extra in ("nonTarget", "all_variants"):
    if extra in records:
        r = records[extra]
        print(f"  {extra:<10} {r['missing']:>6} of {r['unique_variants']:>6}  ({r['pct_missing']:>5.1f}%)")

total_missing = len(set().union(*missing_by_sheet.values()))
print("-" * 52)
print(f"  {'UNION':<10} {total_missing:>6} distinct variants unscored across all sheets")

## 4. Are the gaps concentrated in particular genes?

If whole genes were skipped, the fix is different from variants being dropped one at a time
inside genes that are otherwise covered.

In [ ]:
def gene_breakdown(df, scored):
    d = df.dropna(subset=["Variant"]).drop_duplicates("Variant")[["Variant", "GeneSymbol"]].copy()
    d["found"] = d["Variant"].isin(scored)
    g = d.groupby("GeneSymbol")["found"].agg(n="size", found="sum")
    g["missing"] = g["n"] - g["found"]
    g["frac_missing"] = g["missing"] / g["n"]
    return g


for d in DISEASES:
    if d not in sheets or "GeneSymbol" not in sheets[d].columns:
        continue
    g = gene_breakdown(sheets[d], scored_variants)
    full = g[g.frac_missing == 1]
    part = g[(g.frac_missing > 0) & (g.frac_missing < 1)]
    print(f"{d}: {len(g)} genes | fully unscored {len(full)} ({full['n'].sum()} variants) "
          f"| partially unscored {len(part)} ({part['missing'].sum()} variants) "
          f"| fully covered {(g.frac_missing == 0).sum()}")

## 5. Why are they missing? Walk back to the annotation step

Two possibilities worth separating:

1. **Key mismatch** — the variants are present but written differently (allele order, chr prefix).
2. **Genuinely never processed** — they never entered the pipeline.

The pipeline order is `variant_annotations.tsv` (VEP) → MutPred2 scoring. If a missing variant
is absent from `variant_annotations.tsv` as well, the gap originates *upstream* of scoring.

In [ ]:
# --- possibility 1: key mismatch -------------------------------------------
all_missing = set().union(*missing_by_sheet.values())


def chr_pos(v):
    a = v.split("_")
    return f"{a[0]}_{a[1]}"


def swap_alleles(v):
    head, alleles = v.rsplit("_", 1)
    ref, alt = alleles.split("/")
    return f"{head}_{alt}/{ref}"


scored_pos = {chr_pos(v) for v in scored_variants}

same_pos = {v for v in all_missing if chr_pos(v) in scored_pos}
swapped  = {v for v in all_missing if swap_alleles(v) in scored_variants}

print(f"missing variants sharing a chr_pos with a scored variant : {len(same_pos)} / {len(all_missing)}")
print(f"missing variants explained by ref/alt swap               : {len(swapped)}")
print("  -> a near-zero swap count means this is NOT a key-format artifact")

In [ ]:
import pandas as pd

from ExTSP.config import VariantAnnotations_file

# --- possibility 2: never annotated ----------------------------------------
va = pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
annotated_variants = set(va["Variant"].dropna().unique())

print(f"variants in variant_annotations.tsv : {len(annotated_variants)}")
print(f"variants with a MutPred2 score      : {len(scored_variants)}")
print(f"annotated but unscored              : {len(annotated_variants - scored_variants)}")
print()

rows = []
for name, miss in missing_by_sheet.items():
    rows.append({
        "sheet": name,
        "missing": len(miss),
        "annotated_but_unscored": len(miss & annotated_variants),
        "never_annotated": len(miss - annotated_variants),
    })

diagnosis = pd.DataFrame(rows).set_index("sheet")
diagnosis

**Reading the table above**

- `annotated_but_unscored` — VEP saw them, MutPred2 did not. Re-run the scoring step.
- `never_annotated` — they never reached VEP at all. The variant extraction has moved on since
  `variant_annotations.tsv` was built, so the annotation step needs re-running first.

Compare file timestamps to confirm the ordering.

In [ ]:
import datetime

from ExTSP.config import (
    ExtractedVariants_file,
    MutPred2_filtered_file,
    VariantAnnotations_file,
)

for f in (VariantAnnotations_file, MutPred2_filtered_file, ExtractedVariants_file):
    ts = datetime.datetime.fromtimestamp(f.stat().st_mtime)
    print(f"{ts:%Y-%m-%d %H:%M}   {f.name}")

## 6. Export the unscored variants

Writes the full list so it can be pushed back through annotation + MutPred2 scoring.

In [ ]:
import pandas as pd

out_rows = []
for name, miss in missing_by_sheet.items():
    for v in sorted(miss):
        out_rows.append({
            "Variant": v,
            "sheet": name,
            "never_annotated": v not in annotated_variants,
        })

missing_df = pd.DataFrame(out_rows)

out_path = AI_GENERATED_DIR / "variants_missing_mutpred2.tsv"
missing_df.to_csv(out_path, sep="\t", index=False)

print(f"wrote {len(missing_df)} sheet-variant rows "
      f"({missing_df['Variant'].nunique()} distinct variants) to:")
print(f"  {out_path}")
missing_df.head()

## 7. How many of these variants did Tulika's data already score?

Section 5 showed the unscored variants never reached VEP in *our* pipeline. But Tulika's
earlier run covered a different — and for some diseases larger — slice of ClinVar, so some
of them may already have a MutPred2 score there.

Source: `Tulika_data/classification/exTSP_file_expression_ensemble_annotation_{disease}.tsv`.
These files are variant x transcript x tissue, so they are large (CM is ~800 MB) and heavily
duplicated at the variant level. We stream them in chunks and keep only the two columns we
need, reducing each to the set of variants carrying a non-null `MutPred2`.

In [ ]:
import pandas as pd


from pathlib import Path
from ExTSP.config import DATA_DIR

CLASSIFICATION_DIR = Path(str(DATA_DIR)) / "Tulika_data" / "classification"
print("reading from:", CLASSIFICATION_DIR)


def tulika_scored_variants(disease, chunksize=2_000_000):
    """Set of variants with a non-null MutPred2 score in Tulika's classification file."""
    path = CLASSIFICATION_DIR / f"exTSP_file_expression_ensemble_annotation_{disease}.tsv"
    if not path.exists():
        print(f"  {disease}: MISSING {path.name}")
        return None

    scored = set()
    for chunk in pd.read_csv(path, sep="\t", usecols=["Variant", "MutPred2"],
                             chunksize=chunksize):
        scored.update(chunk.loc[chunk["MutPred2"].notna(), "Variant"].dropna().unique())
    return scored


tulika_scored = {}
for d in DISEASES:
    tulika_scored[d] = tulika_scored_variants(d)
    if tulika_scored[d] is not None:
        print(f"  {d:<5} {len(tulika_scored[d]):>6} distinct scored variants")

### Coverage comparison

For each disease sheet:

- `in_new` / `in_tulika` — sheet variants scored in our current `MutPred2_scores.tsv` vs. in Tulika's data
- `recoverable` — of the ones **we** are missing, how many Tulika already has
- `still_missing` — in neither source; these genuinely need rescoring

In [ ]:
import pandas as pd

rows = []
for d in DISEASES:
    tul = tulika_scored.get(d)
    if tul is None:
        continue

    sheet_vars = set(sheets[d]["Variant"].dropna().unique())
    missing_new = sheet_vars - scored_variants

    rows.append({
        "sheet_variants": len(sheet_vars),
        "in_new": len(sheet_vars & scored_variants),
        "in_tulika": len(sheet_vars & tul),
        "missing_from_new": len(missing_new),
        "recoverable": len(missing_new & tul),
        "still_missing": len(missing_new - tul),
        "tulika_file_total": len(tul),
    })

tulika_cmp = pd.DataFrame(rows, index=[d for d in DISEASES if tulika_scored.get(d) is not None])
tulika_cmp.index.name = "disease"
tulika_cmp

In [ ]:
print("Of the variants we cannot score, how many did Tulika already have?")
print("-" * 68)
for d, r in tulika_cmp.iterrows():
    pct = 100 * r["recoverable"] / r["missing_from_new"] if r["missing_from_new"] else 0
    print(f"  {d:<5} missing {r['missing_from_new']:>6}  ->  recoverable {r['recoverable']:>6} "
          f"({pct:>5.1f}%)   still missing {r['still_missing']:>6}")

print("-" * 68)
tot_missing = tulika_cmp["missing_from_new"].sum()
tot_recover = tulika_cmp["recoverable"].sum()
print(f"  {'TOTAL':<5} missing {tot_missing:>6}  ->  recoverable {tot_recover:>6} "
      f"({100 * tot_recover / tot_missing:>5.1f}%)   still missing {tulika_cmp['still_missing'].sum():>6}")

print()
print("Sanity check -- is every variant in Tulika's disease file also in that disease sheet?")
for d in tulika_cmp.index:
    extra = tulika_scored[d] - set(sheets[d]["Variant"].dropna().unique())
    print(f"  {d:<5} in Tulika's file but not in the sheet: {len(extra)}")

### Reading this section

Where `recoverable` is high, those variants do **not** need to go back through MutPred2 — a
score already exists in Tulika's data and could be merged in, exactly as was done for the
old-variant gap-fill in `test.ipynb`.

Where `still_missing` dominates, the variants are absent from both sources and genuinely
require a fresh annotation + scoring run.

The sanity check matters for interpretation: if Tulika's per-disease file contains no variants
outside the corresponding sheet, then `in_tulika` equals her file's full size, and her coverage
of that sheet is simply `tulika_file_total / sheet_variants`.

## 8. Why Tulika's scored variants did not survive the rescoring round

The intended guarantee was: after Tim's rescoring, our score table should contain **at least**
every target variant Tulika had already scored. Section 7 shows it does not — 12,188 are absent.

This section establishes where they were lost. Two candidate mechanisms are tested:

- **A. Key collision** — cell 15 of `test.ipynb` merged on `ID` + `Substitution` rather than
  `Variant`, so Tulika rows could be absorbed by an existing row and never flagged as missing.
- **B. Wrong filter list** — cell 15 filtered Tulika's scores with
  `mp2_old[mp2_old.Variant.isin(old_disease_variants)]`, where `old_disease_variants` came from
  `get_disease_variants()` reading `ExtractedVariants_file`. If that workbook's sheets held the
  **new** variant set rather than the old one, the filter would discard precisely the
  old-only variants the exercise existed to preserve.

In [ ]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, FILTERED_DIR

TULIKA_RAW = Path(str(DATA_DIR)) / "MutPred2_scores" / "mutpred_score_variant_transcript_tulika.tsv"
SJ_RAW     = Path(str(DATA_DIR)) / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"
NEW_WORKBOOK = FILTERED_DIR / "extractedVariants_copy.xlsx"   # the new-variant workbook

# Everything Tulika scored for the 5 diseases, that we do not have.
tulika_gap = {d: (tulika_scored[d] - scored_variants) for d in DISEASES if tulika_scored.get(d)}
all_gap = set().union(*tulika_gap.values())
print(f"Tulika-scored target variants absent from our table: {len(all_gap)}")

# --- Was the gap ever sent to Tim? ----------------------------------------
request = pd.read_csv(Path("mp2_to_rescore2.tsv"), sep="\t") \
    if Path("mp2_to_rescore2.tsv").exists() else None
if request is not None:
    requested = set(request["Variant"].dropna().unique())
    print(f"  of those, actually requested from Tim : {len(all_gap & requested)}")
    print(f"  of those, never requested             : {len(all_gap - requested)}")

In [ ]:
import pandas as pd

# --- Mechanism A: key collision -------------------------------------------
raw_t = pd.read_csv(TULIKA_RAW, sep="\t", low_memory=False) \
          .rename(columns={"Feature": "ID", "mutation": "Variant"})
sj = pd.read_csv(SJ_RAW, sep="\t")
sj_keys = set(zip(sj["ID"], sj["Substitution"]))

g = raw_t[raw_t["Variant"].isin(all_gap)]
gap_keys = set(zip(g["ID"], g["Substitution"]))

print("Mechanism A -- (ID, Substitution) collision with SJ's file")
print(f"  gap variant-transcript keys            : {len(gap_keys)}")
print(f"  already present in SJ's file           : {len(gap_keys & sj_keys)} "
      f"({100 * len(gap_keys & sj_keys) / len(gap_keys):.1f}%)")
print("  -> a near-zero share rules this out as the cause")

In [ ]:
import pandas as pd

# --- Mechanism B: the filter list was the NEW variant set -------------------
xl_new = pd.ExcelFile(NEW_WORKBOOK)

rows = []
for d in DISEASES:
    tul = tulika_scored.get(d)
    if tul is None:
        continue
    new_sheet = set(xl_new.parse(d)["Variant"].dropna().unique()) if d in xl_new.sheet_names else set()
    rows.append({
        "tulika_scored": len(tul),
        "new_sheet": len(new_sheet),
        "predicted_survivors": len(tul & new_sheet),   # what the filter would have kept
        "actual_in_ours": len(tul & scored_variants),  # what we actually ended up with
        "lost": len(tul - scored_variants),
    })

mech_b = pd.DataFrame(rows, index=[d for d in DISEASES if tulika_scored.get(d)])
mech_b.index.name = "disease"
mech_b["delta"] = mech_b["actual_in_ours"] - mech_b["predicted_survivors"]
print("Mechanism B -- filtering Tulika's scores by the NEW variant sheet")
print(mech_b.to_string())
print()
print("If 'predicted_survivors' tracks 'actual_in_ours', the filter list is the cause.")

### What to do about it

These variants were never sent to Tim, so no rescored values exist for them. Tulika's original
scores *do* exist, but those are the scores the rescoring round was created to distrust —
so they should not simply be merged in.

The cell below emits the rescore request that should have been sent originally: every
Tulika-scored target variant we lack, at variant-transcript resolution, carrying her original
score for reference.

In [ ]:
missed_request = raw_t[raw_t["Variant"].isin(all_gap)][
    ["Variant", "ID", "Substitution", "mutpred"]
].rename(columns={"mutpred": "MutPred2_tulika"}).drop_duplicates(
    subset=["Variant", "ID", "Substitution"]
)

disease_of = {}
for d, gap in tulika_gap.items():
    for v in gap:
        disease_of.setdefault(v, []).append(d)
missed_request["diseases"] = missed_request["Variant"].map(
    lambda v: ",".join(sorted(disease_of.get(v, [])))
)

out_path = AI_GENERATED_DIR / "mp2_to_rescore_missed_tulika.tsv"
missed_request.to_csv(out_path, sep="\t", index=False)

print(f"{len(missed_request)} variant-transcript rows "
      f"({missed_request['Variant'].nunique()} distinct variants) written to:")
print(f"  {out_path}")
print()
print(missed_request["diseases"].value_counts().to_string())

## 9. Inventory: was the analysis ever redone correctly?

Section 8 found a filter bug. This section checks whether a **corrected** run exists somewhere
in `data/` — a different rescore list, or a merge that does contain Tulika's variants.

Each candidate file is scored on one question: *what fraction of the 12,185 lost variants does
it contain?* A file at 100% was built after the fix; a file at 0% was built before it.

In [ ]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, MutPred2_filtered_file

SCORE_DIR = Path(str(DATA_DIR)) / "MutPred2_scores"
PROC      = Path(str(DATA_DIR)) / "processed"

lost_variants = set().union(*[(tulika_scored[d] - scored_variants)
                              for d in DISEASES if tulika_scored.get(d)])
print(f"lost variants (Tulika-scored, absent from our table): {len(lost_variants)}\n")

candidates = [
    SCORE_DIR / "MutPred2_for_filtered_variants_new_plus.tsv",
    SCORE_DIR / "mutpred_for_filtered_variants_old_plus.tsv",
    SCORE_DIR / "MutPred2_for_filtered_variants_new_plus_error.tsv",
    SCORE_DIR / "mutpred_for_filtered_variants_old_plus_error.tsv",
    PROC / "MutPred2_processed_with_IsoPath.tsv",
    PROC / "MP2_Iso_obsolete" / "MutPred2_processed_with_IsoPath_new.tsv",
    PROC / "MP2_Iso_obsolete" / "MutPred2_processed_with_IsoPath_new_plus.tsv",
    MutPred2_filtered_file,
]

rows = []
for path in candidates:
    if not path.exists():
        continue
    df = pd.read_csv(path, sep="\t", low_memory=False)
    if "Variant" not in df.columns:
        continue
    present = set(df.loc[df["MutPred2"].notna(), "Variant"].dropna().unique())
    rows.append({
        "file": path.name,
        "mtime": pd.Timestamp(path.stat().st_mtime, unit="s").strftime("%Y-%m-%d"),
        "rows": len(df),
        "lost_covered": len(lost_variants & present),
        "pct": round(100 * len(lost_variants & present) / len(lost_variants), 1),
    })

inventory = pd.DataFrame(rows).sort_values("pct", ascending=False).set_index("file")
inventory

In [ ]:
from pathlib import Path

import pandas as pd

# Rescore REQUEST lists -- which variants were each time asked to be rescored?
requests = {
    "mp2_to_rescore2.tsv":     Path("mp2_to_rescore2.tsv"),
    "variants_to_rescore.tsv": Path("variants_to_rescore.tsv"),
}

for name, path in requests.items():
    if not path.exists():
        print(f"{name}: not found next to this notebook")
        continue
    if name == "variants_to_rescore.tsv":      # headerless, single column
        v = set(pd.read_csv(path, sep="\t", header=None)[0].dropna().unique())
    else:
        v = set(pd.read_csv(path, sep="\t")["Variant"].dropna().unique())
    print(f"{name:<26} {len(v):>6} variants | covers {len(lost_variants & v):>6} "
          f"of the lost set ({100 * len(lost_variants & v) / len(lost_variants):.1f}%)")

In [ ]:
import numpy as np
import pandas as pd

# Did any rescored VALUES ever come back for the corrected list?
# If they did, scores for lost variants would differ from Tulika's originals.
raw_idx = raw_t.drop_duplicates(subset=["Variant", "ID", "Substitution"]) \
               .set_index(["Variant", "ID", "Substitution"])["mutpred"]

for path in [SCORE_DIR / "MutPred2_for_filtered_variants_new_plus.tsv",
             SCORE_DIR / "mutpred_for_filtered_variants_old_plus.tsv"]:
    df = pd.read_csv(path, sep="\t")
    sub = df[df["Variant"].isin(lost_variants)][["Variant", "ID", "Substitution", "MutPred2"]].dropna()
    ref = raw_idx.reindex(pd.MultiIndex.from_frame(sub[["Variant", "ID", "Substitution"]])).values
    joined = np.sum(~np.isnan(ref))
    differing = int(np.nansum(np.abs(sub["MutPred2"].values - ref) > 1e-6))
    print(f"{path.name}")
    print(f"   lost-variant rows matched to Tulika's originals: {joined}")
    print(f"   of those, values that DIFFER (i.e. rescored)   : {differing}")

print()
print("differing == 0 means no rescored values ever came back for the corrected list.")

### Verdict

The corrected run does exist, but it stopped one step short:

- `variants_to_rescore.tsv` is the **corrected request** — it covers 100% of the lost variants.
- The Jul-22 `*_plus.tsv` files are the **corrected merge** — they contain all of them.
- The Jul-21 `*_plus_error.tsv` files are the buggy pair, and `MutPred2_scores.tsv` was built
  from those, one day *before* the fix landed.
- No file anywhere carries rescored *values* for the corrected list.

Note the corrected `new_plus` is not a drop-in replacement for `MutPred2_scores.tsv`: its
`Variant` column is populated only for Tulika's rows (~130k nulls), because cell 15 merged on
`ID` + `Substitution`. The `Variant` back-fill from `VariantAnnotations_file` that cell 8 of
`test.ipynb` performs still has to be re-applied on top of it.

## 10. Independent rebuild of the "variants my set missed" list

Built from scratch, without reference to any previously generated list, then compared
against the Jul-22 output.

Definition used: a variant counts as **missed** if it appears in a disease sheet of
`extractedVariants.xlsx` but has no MutPred2 score in **my own** scored file,
`MutPred2_for_filtered_variants_SJ.tsv`.

That file is keyed by `ID` (ENST) + `Substitution` and has no `Variant` column, so it is
mapped to genomic variants through `variant_annotations.tsv` — which covers exactly the
new variant set, making it the correct mapping source for my own scores.

In [ ]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, VariantAnnotations_file

SJ_FILE = Path(str(DATA_DIR)) / "MutPred2_scores" / "MutPred2_for_filtered_variants_SJ.tsv"

# --- my own scored set, mapped to genomic variants -------------------------
sj = pd.read_csv(SJ_FILE, sep="\t")
sj = sj[sj["MutPred2"].notna()]

ann = pd.read_csv(VariantAnnotations_file, sep="\t", low_memory=False)
ann = ann[["ENST", "aa_sub", "Variant"]].dropna().drop_duplicates()

sj_mapped = ann.merge(sj, left_on=["ENST", "aa_sub"],
                          right_on=["ID", "Substitution"], how="inner")
SJ_SCORED = set(sj_mapped["Variant"].unique())

print(f"my scored variant-transcript rows : {len(sj)}")
print(f"annotation rows                   : {len(ann)}")
print(f"my scored variants (mapped)       : {len(SJ_SCORED)}")

In [ ]:
# --- what the sheets contain that my set does not --------------------------
my_missing = {}
print(f"{'sheet':<11}{'variants':>9}{'in mine':>9}{'MISSED':>9}")
for s in DISEASES + ["nonTarget"]:
    v = set(sheets[s]["Variant"].dropna().unique())
    my_missing[s] = v - SJ_SCORED
    print(f"{s:<11}{len(v):>9}{len(v & SJ_SCORED):>9}{len(my_missing[s]):>9}")

MY_LIST     = set().union(*[my_missing[s] for s in DISEASES])   # 5 disease sheets
MY_LIST_ALL = set().union(*my_missing.values())                 # + nonTarget

print(f"\nMY_LIST (disease sheets only) : {len(MY_LIST)}")
print(f"MY_LIST_ALL (incl. nonTarget) : {len(MY_LIST_ALL)}")

### Compare against what was generated on Jul 22

The Jul-22 `_plus` files are score tables, not lists. The comparable quantity is the set of
variants they add **on top of** my own scored set — i.e. what the corrected merge contributed.

In [ ]:
from pathlib import Path

import pandas as pd

def compare(name, other, mine):
    print(f"\n--- {name} ({len(other)})  vs  MY_LIST ({len(mine)}) ---")
    print(f"  identical      : {mine == other}")
    print(f"  in both        : {len(mine & other)}")
    print(f"  only in mine   : {len(mine - other)}")
    print(f"  only in theirs : {len(other - mine)}")
    print(f"  Jaccard        : {len(mine & other) / len(mine | other):.4f}")


for fname in ["MutPred2_for_filtered_variants_new_plus.tsv",
              "mutpred_for_filtered_variants_old_plus.tsv"]:
    df = pd.read_csv(SCORE_DIR / fname, sep="\t")
    added = set(df.loc[df["MutPred2"].notna(), "Variant"].dropna().unique()) - SJ_SCORED
    compare(f"{fname} [added over my set]", added, MY_LIST)

rescore_list = set(pd.read_csv(Path("variants_to_rescore.tsv"), sep="\t",
                               header=None)[0].dropna().unique())
compare("variants_to_rescore.tsv", rescore_list, MY_LIST)

In [ ]:
import pandas as pd

# --- what accounts for the difference? -------------------------------------
df = pd.read_csv(SCORE_DIR / "MutPred2_for_filtered_variants_new_plus.tsv", sep="\t")
jul22_added = set(df.loc[df["MutPred2"].notna(), "Variant"].dropna().unique()) - SJ_SCORED
delta = MY_LIST - jul22_added

tulika_raw_variants = set(raw_t.loc[raw_t["mutpred"].notna(), "Variant"].dropna().unique())
print(f"in my list but not in the Jul-22 merge : {len(delta)}")
print(f"  present somewhere in Tulika's raw    : {len(delta & tulika_raw_variants)}")
print(f"  scored by NEITHER of us              : {len(delta - tulika_raw_variants)}")
print()
for s in DISEASES:
    print(f"  {s:<6} missed={len(my_missing[s]):>6}  in Jul-22={len(my_missing[s] & jul22_added):>6}"
          f"  delta={len(my_missing[s] - jul22_added):>6}")

In [ ]:
import pandas as pd

# --- emit the independent list ---------------------------------------------
rows = [{"Variant": v,
         "sheets": ",".join(sorted(s for s in DISEASES if v in my_missing[s])),
         "in_jul22_merge": v in jul22_added}
        for v in sorted(MY_LIST)]
my_list_df = pd.DataFrame(rows)

out = AI_GENERATED_DIR / "variants_missed_by_my_set.tsv"
my_list_df.to_csv(out, sep="\t", index=False)

# bare one-column list, the format Tim was sent previously
bare = AI_GENERATED_DIR / "variants_to_rescore_independent.txt"
pd.Series(sorted(MY_LIST)).to_csv(bare, sep="\t", index=False, header=False)

print(f"{len(my_list_df)} variants written to:\n  {out}\n  {bare}")
my_list_df.head()

### Result

The two lists are **not identical**. The Jul-22 merge contributes a strict subset: every one of
its variants is in the independent list, and it adds nothing the independent list lacks.

The shortfall is the set of variants that **neither** my scoring run nor Tulika's ever covered —
a merge of the two sources cannot surface them by construction, which is why they were never
queued for rescoring.

## 11. Correction: most of the "missing" variants are not missense

The nonTarget result in section 10 is the tell. nonTarget variants came from the new data and
were scored directly, so ~5,000 of them being unscored makes no sense — unless they are not
MutPred2-scorable in the first place.

MutPred2 scores **missense substitutions only**. The ClinVar `Name` column carries the protein
consequence (`p.Gln437Ter`, `p.Arg3681Ter`, ...), so it can be classified directly. Anything
ending in `Ter` is a stop-gain, `=` is synonymous — neither yields an `aa_sub`, so neither
appears in `variant_annotations.tsv` and neither can ever be scored.

In [ ]:
import re

def consequence(name):
    """Classify a ClinVar Name string by protein consequence."""
    if not isinstance(name, str):
        return "no_name"
    m = re.search(r"\(p\.([A-Za-z]{3})(\d+)([A-Za-z]{3}|Ter|=)\)", name)
    if not m:
        return "unparsed"
    ref, pos, alt = m.group(1), m.group(2), m.group(3)
    if alt == "Ter":
        return "nonsense"
    if alt == "=":
        return "synonymous"
    if ref == "Met" and pos == "1":
        return "start_lost"
    return "missense"


print(f"{'sheet':<11}{'missed':>8}{'missense':>10}{'nonsense':>10}{'other':>8}")
missense_missing = {}
for s in DISEASES + ["nonTarget"]:
    df = sheets[s].drop_duplicates("Variant")
    if "Name" not in df.columns:          # ASD sheet carries no consequence column
        missense_missing[s] = my_missing[s]
        print(f"{s:<11}{len(my_missing[s]):>8}{'n/a -- no Name column':>30}")
        continue
    sub = df[df["Variant"].isin(my_missing[s])]
    cls = sub["Name"].map(consequence)
    missense_missing[s] = set(sub.loc[cls == "missense", "Variant"])
    n = len(sub)
    print(f"{s:<11}{n:>8}{(cls == 'missense').sum():>10}"
          f"{(cls == 'nonsense').sum():>10}{n - (cls.isin(['missense','nonsense'])).sum():>8}")

In [ ]:
MY_LIST_MISSENSE = set().union(*[missense_missing[s] for s in DISEASES])

print(f"missense-only missed, disease sheets : {len(MY_LIST_MISSENSE):>6}   (raw was {len(MY_LIST)})")
print(f"missense-only missed, nonTarget      : {len(missense_missing['nonTarget']):>6}   "
      f"(raw was {len(my_missing['nonTarget'])})")
print()
print("-> nonTarget is essentially complete, as expected: its unscored variants are")
print("   overwhelmingly stop-gains that MutPred2 cannot score.")

### Re-run the Jul-22 comparison on missense only

In [ ]:
compare("Jul-22 new_plus [added over my set]", jul22_added, MY_LIST_MISSENSE)

only_mine   = MY_LIST_MISSENSE - jul22_added
only_theirs = jul22_added - MY_LIST_MISSENSE

print("\nonly-in-mine, by sheet:")
for s in DISEASES:
    print(f"  {s:<6} {len(only_mine & missense_missing[s]):>5}")
print(f"\n  scored by neither of us : {len(only_mine - tulika_raw_variants)}")
print(f"  present in Tulika's raw : {len(only_mine & tulika_raw_variants)}")

print(f"\nonly-in-theirs ({len(only_theirs)}):")
for s in DISEASES:
    df = sheets[s]
    cols = [c for c in ("Variant", "GeneSymbol", "Name") if c in df.columns]
    hit = df[df["Variant"].isin(only_theirs)][cols].drop_duplicates()
    if len(hit):
        print(f"  [{s}]")
        print(hit.to_string(index=False))

In [ ]:
import pandas as pd

# --- final list: missense variants my set missed ---------------------------
rows = [{"Variant": v,
         "sheets": ",".join(sorted(s for s in DISEASES if v in missense_missing[s])),
         "in_jul22_merge": v in jul22_added,
         "scored_by_neither": v not in tulika_raw_variants}
        for v in sorted(MY_LIST_MISSENSE)]
final_df = pd.DataFrame(rows)

out  = AI_GENERATED_DIR / "variants_missed_missense.tsv"
bare = AI_GENERATED_DIR / "variants_to_rescore_missense.txt"
final_df.to_csv(out, sep="\t", index=False)
pd.Series(sorted(MY_LIST_MISSENSE)).to_csv(bare, sep="\t", index=False, header=False)

print(f"{len(final_df)} missense variants written to:\n  {out}\n  {bare}")
print()
print(final_df["scored_by_neither"].value_counts().to_string())

### Conclusion

Restricted to missense — the only thing MutPred2 can score — the independently derived list and
the Jul-22 merge agree almost exactly: **12,188 shared**, 996 extra in the independent list,
1 extra in the Jul-22 merge.

- The **996** are variants neither scoring run covered (973 of them), so a merge of the two
  sources could not have produced them. 923 are ASD.
- The **1** is `X_31126647_T/A` (DMD, `p.Arg3681Ter`) — a stop-gain that Tulika's pipeline
  scored anyway, which is itself a symptom of the extraction problems that motivated rescoring.

**Caveat:** the ASD sheet has no `Name`/consequence column, so its 927 variants could not be
filtered this way and are carried through unclassified. If ASD follows the pattern of the other
sheets, some fraction are not missense and the true figure is below 13,184.

## 12. Build the rescore request for Tim

Three steps: confirm the workbook's disease sheets really are Tulika's target lists, work out
what is missing from the SJ scoring run, and assemble a single table with positive controls.

Non-missense variants are **kept** here — Tim decides what is scorable.

### 12a. Confirm the disease sheets match Tulika's target variant lists

In [ ]:
from pathlib import Path

import pandas as pd

from ExTSP.config import DATA_DIR, ExtractedVariants_file

DISEASES = ["CM", "PCD", "PH", "PKD", "ASD"]
TARGET_DIR = Path(str(DATA_DIR)) / "Tulika_data" / "targetVariants"

# Read the sheets here so this cell runs on its own; reuse them if already loaded.
if "sheets" not in dir():
    xl = pd.ExcelFile(ExtractedVariants_file)
    sheets = {name: xl.parse(name) for name in xl.sheet_names}

print(f"{'sheet':<7}{'excel':>8}{'target':>8}{'identical':>11}{'only_excel':>12}{'only_target':>13}")
all_identical = True
for d in DISEASES:
    sheet_vars = set(sheets[d]["Variant"].dropna().unique())
    tgt = pd.read_csv(TARGET_DIR / f"High_Confident_SNP_mutations_{d}.tsv", sep="\t")
    tgt_vars = set(tgt["Variant_ID"].dropna().unique())
    same = sheet_vars == tgt_vars
    all_identical &= same
    print(f"{d:<7}{len(sheet_vars):>8}{len(tgt_vars):>8}{str(same):>11}"
          f"{len(sheet_vars - tgt_vars):>12}{len(tgt_vars - sheet_vars):>13}")

print()
print("All disease sheets identical to Tulika's target lists:", all_identical)
assert all_identical, "a disease sheet no longer matches its Tulika target list"


### 12b. Master table of variants missing from the SJ scoring run

Every target and nonTarget variant in the workbook is checked against the SJ MutPred2 file,
then against Tulika's MutPred2 file. Variants absent from SJ form the request; the `Tulika`
column records whether a score exists on her side, so rows with `Tulika == False` are the ones
neither of us ever scored.

In [ ]:
SHEETS_FOR_REQUEST = DISEASES + ["nonTarget"]

# Provenance: which sheets each variant appears in.
sheet_membership = {}
for s in SHEETS_FOR_REQUEST:
    for v in sheets[s]["Variant"].dropna().unique():
        sheet_membership.setdefault(v, []).append(s)

all_workbook_variants = set(sheet_membership)
missing_from_sj = all_workbook_variants - SJ_SCORED

print(f"workbook variants (target + nonTarget) : {len(all_workbook_variants)}")
print(f"  scored by SJ                         : {len(all_workbook_variants & SJ_SCORED)}")
print(f"  MISSING from SJ                      : {len(missing_from_sj)}")
print(f"     of those, Tulika has a score      : {len(missing_from_sj & tulika_raw_variants)}")
print(f"     of those, neither of us has one   : {len(missing_from_sj - tulika_raw_variants)}")

In [ ]:
import pandas as pd

# Consequence is informational only -- nothing is filtered out on it.
name_lookup = {}
for s in SHEETS_FOR_REQUEST:
    df = sheets[s]
    if "Name" in df.columns:
        for v, n in zip(df["Variant"], df["Name"]):
            if v not in name_lookup:
                name_lookup[v] = n

request_rows = [{
    "Variant": v,
    "sheets": ",".join(sheet_membership[v]),
    "Tulika": v in tulika_raw_variants,
    "SJ": v in SJ_SCORED,
    "missing": True,
    "consequence": consequence(name_lookup.get(v)),
} for v in sorted(missing_from_sj)]

master = pd.DataFrame(request_rows)
print(f"master table (missing block): {len(master)} rows\n")
print(master.groupby(["Tulika", "SJ"]).size().rename("n").to_string())
print()
print(master["consequence"].value_counts().to_string())

### 12c. Add 1,000 positive controls

A random sample of variants SJ already scored, flagged `missing = False`. Re-scoring these
should reproduce the existing values; any drift is a red flag on the run itself.

The seed is fixed so the same controls come back on a re-run.

In [ ]:
import pandas as pd

SEED, N_CONTROLS = 42, 1000

control_pool = sorted(all_workbook_variants & SJ_SCORED)
controls = pd.Series(control_pool).sample(n=N_CONTROLS, random_state=SEED)

control_rows = [{
    "Variant": v,
    "sheets": ",".join(sheet_membership[v]),
    "Tulika": v in tulika_raw_variants,
    "SJ": True,
    "missing": False,
    "consequence": consequence(name_lookup.get(v)),
} for v in controls]

master = pd.concat([master, pd.DataFrame(control_rows)], ignore_index=True)

assert master["Variant"].is_unique, "controls overlap the missing block"
assert not master.loc[master["missing"], "SJ"].any(), "a missing row is marked as SJ-scored"
assert master.loc[~master["missing"], "SJ"].all(), "a control row is not SJ-scored"

print(f"master table: {len(master)} rows "
      f"({(master['missing']).sum()} to score, {(~master['missing']).sum()} controls)\n")
print(master.groupby(["missing", "Tulika"]).size().rename("n").to_string())

In [ ]:
out  = AI_GENERATED_DIR / "mp2_request_for_tim.tsv"
bare = AI_GENERATED_DIR / "mp2_request_for_tim_variants.txt"

master.to_csv(out, sep="\t", index=False)
master["Variant"].to_csv(bare, sep="\t", index=False, header=False)

print(f"{len(master)} variants written to:\n  {out}\n  {bare}")
master.head(10)